# 🌉 Module 06: Bridging Sync and Async (The Golden Rule)

In real-world engineering, 100% pure async code is rare. You will inevitably interact with:
- Synchronous AWS SDKs (`boto3`)
- Heavy data science libraries (`pandas`, `numpy`, `scipy`)
- CPU-intensive image resizing (`Pillow` / `opencv`)
- Synchronous legacy SQL databases

### What We Will Master:
1. **The Golden Rule**: *Never block the event loop*.
2. **`asyncio.to_thread`**: Offloading blocking I/O to Python's default threadpool.
3. **`loop.run_in_executor` with `ProcessPoolExecutor`**: Bypassing the Python GIL for true CPU parallelism.
4. **Calling Async from Sync**: How Django uses `asgiref.sync.async_to_sync` without crashing.
5. **Why `asyncio.run()` inside an active loop raises `RuntimeError`**.

In [1]:
import asyncio
import time

def heavy_blocking_pandas_work(n_rows: int = 10_000_000) -> str:
    print(f"[{time.strftime('%X')}] 📊 Starting heavy pandas computation in separate OS thread...")
    time.sleep(1.0) # Simulates heavy CPU/file sync parsing
    print(f"[{time.strftime('%X')}] ✅ Heavy pandas computation completed in background thread!")
    return f"{n_rows:,} rows aggregated."

async def responsive_api_health_check():
    await asyncio.sleep(0.05)
    print(f"[{time.strftime('%X')}] 💓 Health check responded in 0.05s! (Event loop remained 100% free!)")

print("🚀 Launching CPU/Disk blocking Pandas workload alongside responsive health check...")
pandas_task = asyncio.to_thread(heavy_blocking_pandas_work, 10_000_000)
results = await asyncio.gather(pandas_task, responsive_api_health_check())
print("Result:", results[0])

🚀 Launching CPU/Disk blocking Pandas workload alongside responsive health check...
[08:30:01] 📊 Starting heavy pandas computation in separate OS thread...
[08:30:01] 💓 Health check responded in 0.05s! (Event loop remained 100% free!)
[08:30:02] ✅ Heavy pandas computation completed in background thread!
Result: 10,000,000 rows aggregated.


---
## 2. Bypassing the Python GIL: `ProcessPoolExecutor`

For heavy CPU math (encryption, image manipulation, ML inference), `asyncio.to_thread` is still bounded by the Python **GIL (Global Interpreter Lock)**.
To achieve **true parallel computation across all CPU cores**, delegate to a separate process pool using `loop.run_in_executor`.

In [2]:
from concurrent.futures import ProcessPoolExecutor

def cpu_crypto_hash(n: int) -> int:
    return sum(i * i for i in range(n))

loop = asyncio.get_running_loop()
# In production, create this pool once during app startup
with ProcessPoolExecutor(max_workers=2) as process_pool:
    print("🚀 Delegating CPU crypto hashing to ProcessPoolExecutor across separate CPU cores...")
    f1 = loop.run_in_executor(process_pool, cpu_crypto_hash, 10_000_000)
    f2 = loop.run_in_executor(process_pool, cpu_crypto_hash, 10_000_000)
    res1, res2 = await asyncio.gather(f1, f2)
    print(f"Core 1 calculated hash: {res1}")
    print(f"Core 2 calculated hash: {res2}")
    print("✅ Multi-core calculation completed without blocking the async event loop!")

🚀 Delegating CPU crypto hashing to ProcessPoolExecutor across separate CPU cores...
Core 1 calculated hash: 333333283333335000000
Core 2 calculated hash: 333333283333335000000
✅ Multi-core calculation completed without blocking the async event loop!
